### Oppgave 1

In [1]:
"""Temperature class supporting Celsius, Fahrenheit and Kelvin."""


class Temperature:
    """A temperature with a value and a unit ('C', 'F' or 'K')."""

    VALID_UNITS = ("C", "F", "K")
    TOL = 1e-9

    def __init__(self, value: float, unit: str = "C"):
        self._unit = self._validate_unit(unit)
        self._value = float(value)
        if self.to_kelvin() < -self.TOL:
            raise ValueError("Temperature cannot be below absolute zero (0 K).")

    @classmethod
    def _validate_unit(cls, unit: str) -> str:
        """Return the unit in upper case, or raise ValueError if invalid."""
        if not isinstance(unit, str) or unit.upper() not in cls.VALID_UNITS:
            raise ValueError(f"Invalid unit: {unit!r}. Use 'C', 'F' or 'K'.")
        return unit.upper()

    #  Read-only attributes 
    @property
    def value(self) -> float:
        return self._value

    @property
    def unit(self) -> str:
        return self._unit

    #  Conversions 
    def to_celsius(self) -> float:
        if self._unit == "C":
            return self._value
        if self._unit == "K":
            return self._value - 273.15
        return (self._value - 32) * 5 / 9  # from Fahrenheit

    def to_fahrenheit(self) -> float:
        if self._unit == "F":
            return self._value
        if self._unit == "C":
            return self._value * 9 / 5 + 32
        return (self._value - 273.15) * 9 / 5 + 32  # from Kelvin

    def to_kelvin(self) -> float:
        if self._unit == "K":
            return self._value
        if self._unit == "C":
            return self._value + 273.15
        return (self._value - 32) * 5 / 9 + 273.15  # from Fahrenheit

    def convert_to(self, target_unit: str) -> "Temperature":
        target = self._validate_unit(target_unit)
        converters = {
            "C": self.to_celsius,
            "F": self.to_fahrenheit,
            "K": self.to_kelvin,
        }
        return Temperature(converters[target](), target)

    # String representations
    def __repr__(self) -> str:
        return f"Temperature({self._value!r}, {self._unit!r})"

    def __str__(self) -> str:
        return f"{self._value:.2f} °{self._unit}"

    # Comparisons (done in Kelvin, with tolerance)
    def __eq__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return abs(self.to_kelvin() - other.to_kelvin()) < self.TOL

    def __ne__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return not self == other

    def __lt__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() < other.to_kelvin() and not self == other

    def __le__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() < other.to_kelvin() or self == other

    def __gt__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() > other.to_kelvin() and not self == other

    def __ge__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() > other.to_kelvin() or self == other

### Task 1 - test

In [2]:
"""Tests for the Temperature class. Run with: python -m pytest -v test_temperature.py"""

import pytest

from temperature import Temperature


# ---------- 1. Initialization ----------
def test_default_unit():
    assert Temperature(20).unit == "C"


def test_case_insensitive_unit():
    assert Temperature(20, "f").unit == "F"


def test_read_only_attributes():
    t = Temperature(20, "C")
    with pytest.raises(AttributeError):
        t.value = 30
    with pytest.raises(AttributeError):
        t.unit = "K"


# ---------- 2. Validation ----------
def test_invalid_unit():
    with pytest.raises(ValueError):
        Temperature(10, "X")


def test_below_absolute_zero():
    with pytest.raises(ValueError):
        Temperature(-274, "C")
    with pytest.raises(ValueError):
        Temperature(-460, "F")
    with pytest.raises(ValueError):
        Temperature(-1, "K")


# ---------- 3. Conversions ----------
@pytest.mark.parametrize("c, f, k", [
    (-273.15, -459.67, 0.0),   # absolute zero
    (-40, -40, 233.15),        # Celsius-Fahrenheit crossover
    (0, 32, 273.15),           # freezing point of water
    (37, 98.6, 310.15),        # human body temperature
    (100, 212, 373.15),        # boiling point of water
])
def test_conversions(c, f, k):
    t = Temperature(c, "C")
    assert t.to_fahrenheit() == pytest.approx(f)
    assert t.to_kelvin() == pytest.approx(k)
    assert Temperature(f, "F").to_celsius() == pytest.approx(c)
    assert Temperature(k, "K").to_celsius() == pytest.approx(c)


# ---------- 4. Round-trip conversion ----------
def test_round_trip():
    original = Temperature(25, "C")
    back = original.convert_to("F").convert_to("K").convert_to("C")

    assert back == original


    # ---------- 5. Cross-unit comparisons ----------
    def test_cross_unit_equality():
        assert Temperature(0, "C") == Temperature(32, "F")
    
    
    def test_cross_unit_ordering():
        assert Temperature(0, "C") < Temperature(274, "K")
        assert Temperature(50, "F") <= Temperature(10, "C")
        assert Temperature(100, "C") > Temperature(200, "F")
        assert Temperature(373.15, "K") >= Temperature(212, "F")
    
    
    # ---------- 6. Formatting ----------
    def test_str():
        assert str(Temperature(25, "C")) == "25.00 °C"
    
    
    def test_repr():
        assert repr(Temperature(25, "C")) == "Temperature(25.0, 'C')"


In [3]:
import sys
!{sys.executable} -m pytest -v test_temperature.py

============================= test session starts ==============================
platform darwin -- Python 3.9.6, pytest-8.4.2, pluggy-1.6.0 -- /Users/markusvalland/Downloads/INF_201/deliverable_4/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/markusvalland/Downloads/INF_201/deliverable_4
plugins: anyio-4.12.1
collected 15 items                                                             

test_temperature.py::test_default_unit PASSED                            [  6%]
test_temperature.py::test_case_insensitive_unit PASSED                   [ 13%]
test_temperature.py::test_read_only_attributes PASSED                    [ 20%]
test_temperature.py::test_invalid_unit PASSED                            [ 26%]
test_temperature.py::test_below_absolute_zero PASSED                     [ 33%]
test_temperature.py::test_conversions[-273.15--459.67-0.0] PASSED        [ 40%]
test_temperature.py::test_conversions[-40--40-233.15] PASSED             [ 46%]
test_temperature.py::test_conversions